# Phase 5: GAO Validation Track

**Purpose:** Extract weapon system data from GAO PDFs using Gemini API, link to FPDS contracts, and cross-validate classifier predictions.

**Prerequisites:**
- `GEMINI_API_KEY` environment variable set
- 22 GAO Weapon Systems Annual Assessment PDFs downloaded

**Workflow:**
1. Download GAO PDFs (if not present)
2. Chunk PDFs into 5-10 page sections
3. Run Gemini batch extraction
4. Parse and clean results
5. Link GAO programs to FPDS contracts
6. Cross-validate best classifier on linked subset

**Key Outputs:**
- `data/raw/gao_reports/` — Downloaded PDFs
- `data/interim/gao_chunks/` — Chunked PDFs
- `data/interim/gao_results/` — Raw Gemini JSON responses
- `data/processed/gao_validation_set.csv` — Parsed GAO data
- `data/processed/gao_fpds_linked.csv` — Linked GAO-FPDS records

In [ ]:
import sys
import importlib.util

print('Python executable:', sys.executable)
print('Python version:', sys.version)
print('\nPython paths:')
for p in sys.path[:5]:
    print(' ', p)

# Check for optional python-dotenv without hard import
if importlib.util.find_spec('dotenv') is not None:
    print('\n[OK] dotenv package is available')
else:
    print('\n[WARN] dotenv package not found')
    print('Run in terminal (matching this Python):')
    print(f'  {sys.executable} -m pip install python-dotenv')

Python executable: c:\Users\sarme\anaconda3\envs\contracts\python.exe
Python version: 3.11.15 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:12:15) [MSC v.1942 64 bit (AMD64)]

Python paths:
  C:\Users\sarme\IS392Final\IS392\scripts
  ../scripts
  ../scripts
  ../scripts
  c:\Users\sarme\anaconda3\envs\contracts\python311.zip

[OK] dotenv package is available


## 1. Environment Setup

In [29]:
import os
import sys
import pandas as pd
from pathlib import Path

def load_env_file(env_file):
    """Lightweight .env loader to avoid external dependency at runtime."""
    loaded = 0
    if not env_file.exists():
        return loaded
    for raw_line in env_file.read_text(encoding='utf-8', errors='ignore').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        if key and key not in os.environ:
            os.environ[key] = value
            loaded += 1
    return loaded

# Detect project root robustly
cwd = Path.cwd().resolve()
PROJECT_ROOT = None
for base in [cwd] + list(cwd.parents):
    if (base / 'scripts').exists() and (base / 'data').exists():
        PROJECT_ROOT = base
        break
    if (base / 'IS392' / 'scripts').exists() and (base / 'IS392' / 'data').exists():
        PROJECT_ROOT = base / 'IS392'
        break
if PROJECT_ROOT is None:
    PROJECT_ROOT = cwd
    print(f'Warning: Could not detect project root from {cwd}; using cwd.')

# Load environment variables from .env file
env_path = (PROJECT_ROOT / '.env').resolve()
if env_path.exists():
    loaded_count = load_env_file(env_path)
    print(f'Loaded .env from: {env_path} ({loaded_count} vars)')
else:
    print(f'Warning: .env file not found at {env_path}')

# Add scripts directory to path
scripts_dir = str((PROJECT_ROOT / 'scripts').resolve())
if scripts_dir not in sys.path:
    sys.path.insert(0, scripts_dir)

# Configuration
GAO_REPORTS_DIR = str(PROJECT_ROOT / 'data' / 'raw' / 'gao_reports')
GAO_CHUNKS_DIR = str(PROJECT_ROOT / 'data' / 'interim' / 'gao_chunks')
GAO_RESULTS_DIR = str(PROJECT_ROOT / 'data' / 'interim' / 'gao_results')
GAO_VALIDATION_PATH = str(PROJECT_ROOT / 'data' / 'processed' / 'gao_validation_set.csv')
GAO_FPDS_LINKED_PATH = str(PROJECT_ROOT / 'data' / 'processed' / 'gao_fpds_linked.csv')
FPDS_DATA_PATH = str(PROJECT_ROOT / 'data' / 'processed' / 'labeled_contracts.parquet')

# Check Gemini API key
api_key = os.environ.get('GEMINI_API_KEY')
if api_key:
    print(f'GEMINI_API_KEY loaded ({len(api_key)} chars)')
else:
    print('GEMINI_API_KEY not found in .env or environment')
    print('   Add to .env file: GEMINI_API_KEY=your_key_here')

print(f'\nProject root: {PROJECT_ROOT}')
print('\nDirectories:')
print(f'  Reports: {GAO_REPORTS_DIR}')
print(f'  Chunks: {GAO_CHUNKS_DIR}')
print(f'  Results: {GAO_RESULTS_DIR}')

Loaded .env from: C:\Users\sarme\IS392Final\IS392\.env (1 vars)
GEMINI_API_KEY loaded (24 chars)

Project root: C:\Users\sarme\IS392Final\IS392

Directories:
  Reports: C:\Users\sarme\IS392Final\IS392\data\raw\gao_reports
  Chunks: C:\Users\sarme\IS392Final\IS392\data\interim\gao_chunks
  Results: C:\Users\sarme\IS392Final\IS392\data\interim\gao_results


## 2. Step 1: Download GAO PDFs

In [37]:
# Check current status
from pathlib import Path

reports_dir = Path(GAO_REPORTS_DIR)
if reports_dir.exists():
    pdf_files = list(reports_dir.glob('gao_weapons_*.pdf'))
    print(f'GAO PDFs found: {len(pdf_files)}')
    for pdf in sorted(pdf_files):
        size_mb = pdf.stat().st_size / (1024 * 1024)
        print(f'  {pdf.name}: {size_mb:.1f} MB')
else:
    print('No reports directory found.')

print('\nTo download missing PDFs, run from terminal:')
print('  python ../scripts/download_gao_pdfs.py')
print('\nOr manually download from:')
print('  https://www.gao.gov/search?search_api_fulltext=weapon+systems+annual+assessment')

GAO PDFs found: 22
  gao_weapons_01.pdf: 0.0 MB
  gao_weapons_02.pdf: 0.0 MB
  gao_weapons_03.pdf: 0.0 MB
  gao_weapons_04.pdf: 0.0 MB
  gao_weapons_05.pdf: 0.0 MB
  gao_weapons_06.pdf: 0.0 MB
  gao_weapons_07.pdf: 0.0 MB
  gao_weapons_08.pdf: 0.0 MB
  gao_weapons_09.pdf: 0.0 MB
  gao_weapons_10.pdf: 0.0 MB
  gao_weapons_11.pdf: 0.0 MB
  gao_weapons_12.pdf: 0.0 MB
  gao_weapons_13.pdf: 0.0 MB
  gao_weapons_14.pdf: 0.0 MB
  gao_weapons_15.pdf: 0.0 MB
  gao_weapons_16.pdf: 0.0 MB
  gao_weapons_17.pdf: 0.0 MB
  gao_weapons_18.pdf: 0.0 MB
  gao_weapons_19.pdf: 0.0 MB
  gao_weapons_20.pdf: 0.0 MB
  gao_weapons_21.pdf: 0.0 MB
  gao_weapons_22.pdf: 0.0 MB

To download missing PDFs, run from terminal:
  python ../scripts/download_gao_pdfs.py

Or manually download from:
  https://www.gao.gov/search?search_api_fulltext=weapon+systems+annual+assessment


## 3. Step 2: Chunk PDFs

In [38]:
# Check chunk status
chunks_dir = Path(GAO_CHUNKS_DIR)
if chunks_dir.exists():
    chunk_files = list(chunks_dir.glob('gao_*_chunk_*.pdf'))
    print(f'PDF chunks found: {len(chunk_files)}')
else:
    print('No chunks directory found.')
    chunk_files = []

print('\nTo create chunks, run:')
print('  python ../scripts/chunk_gao_pdfs.py')
print('\nThis extracts the appendix section and splits into 5-10 page chunks.')

PDF chunks found: 22

To create chunks, run:
  python ../scripts/chunk_gao_pdfs.py

This extracts the appendix section and splits into 5-10 page chunks.


# Cross-validate best model on GAO-linked subset
from sklearn.metrics import classification_report, roc_auc_score
from pathlib import Path

# Check if linked data exists before loading
linked_path = Path(GAO_FPDS_LINKED_PATH)

if not linked_path.exists():
    print(f'Linked data not found: {linked_path}')
    print('\nPlease run the linking step first:')
    print('  python ../scripts/link_gao_to_fpds.py')
    print('\nNote: This step matches GAO program names to FPDS contract descriptions.')
    linked_df = None
else:
    # Load linked data
    linked_df = pd.read_csv(GAO_FPDS_LINKED_PATH)
    
    if len(linked_df) == 0:
        print('No linked records found. The linking step produced empty results.')
        print('This may indicate:')
        print('  - Low overlap between GAO programs and FPDS contracts')
        print('  - Need to expand GAO PDF set for better coverage')
    elif len(linked_df) < 10:
        print(f'⚠️  Warning: Only {len(linked_df)} linked records found.')
        print('Validation will be limited with this small sample.')
        print(f'\nProceeding with {len(linked_df)} records...')
    else:
        print(f'✓ Validating on {len(linked_df)} linked contracts...')

# Proceed with validation if we have linked data
if linked_df is not None and len(linked_df) > 0:
    # Load FPDS data with predictions (from Phase 3)
    fpds_df = pd.read_parquet(FPDS_DATA_PATH)
    
    # Merge linked GAO data with FPDS
    linked_with_fpds = linked_df.merge(
        fpds_df[['piid', 'over_budget', 'late']], 
        left_on='fpds_piid', 
        right_on='piid',
        how='left'
    )
    
    # Analyze cost overrun predictions
    print('\n' + '=' * 60)
    print('COST OVERRUN CROSS-VALIDATION')
    print('=' * 60)
    
    # GAO-reported cost growth
    gao_cost_overrun = (linked_with_fpds['gao_cost_growth'] > 5).astype(int)
    
    # Compare with FPDS-derived labels
    valid_mask = linked_with_fpds['over_budget'].notna()
    
    if valid_mask.sum() > 0:
        y_true = linked_with_fpds.loc[valid_mask, 'over_budget'].astype(int)
        y_gao = gao_cost_overrun[valid_mask]
        
        print(f'\nLinked contracts with both labels: {valid_mask.sum()}')
        print(f'\nFPDS over_budget rate: {y_true.mean():.1%}')
        print(f'GAO-reported cost overrun (>5% growth): {y_gao.mean():.1%}')
        
        # Agreement statistics
        agreement = (y_true == y_gao).mean()
        print(f'\nLabel agreement: {agreement:.1%}')
        
        # Simple classification metrics
        from sklearn.metrics import confusion_matrix
        cm = confusion_matrix(y_true, y_gao)
        print(f'\nConfusion matrix:')
        print(f'  TN: {cm[0,0]}, FP: {cm[0,1]}')
        print(f'  FN: {cm[1,0]}, TP: {cm[1,1]}')
        
        if cm.shape == (2, 2) and cm[1,1] + cm[0,1] > 0 and cm[1,1] + cm[1,0] > 0:
            precision = cm[1,1] / (cm[1,1] + cm[0,1])
            recall = cm[1,1] / (cm[1,1] + cm[1,0])
            f1 = 2 * (precision * recall) / (precision + recall)
            print(f'\nPrecision: {precision:.2f}, Recall: {recall:.2f}, F1: {f1:.2f}')
    else:
        print('\nNo contracts with valid FPDS labels found.')
        print('This may indicate data processing issues in earlier phases.')
    
    # Analyze schedule delay predictions
    print('\n' + '=' * 60)
    print('SCHEDULE DELAY CROSS-VALIDATION')
    print('=' * 60)
    
    gao_schedule_delay = (linked_with_fpds['gao_schedule_delay'] > 0).astype(int)
    valid_mask_late = linked_with_fpds['late'].notna()
    
    if valid_mask_late.sum() > 0:
        y_true_late = linked_with_fpds.loc[valid_mask_late, 'late'].astype(int)
        y_gao_late = gao_schedule_delay[valid_mask_late]
        
        print(f'\nLinked contracts with both labels: {valid_mask_late.sum()}')
        print(f'\nFPDS late rate: {y_true_late.mean():.1%}')
        print(f'GAO-reported schedule delay: {y_gao_late.mean():.1%}')
        
        agreement_late = (y_true_late == y_gao_late).mean()
        print(f'\nLabel agreement: {agreement_late:.1%}')
    else:
        print('\nNo contracts with valid FPDS late labels found.')

In [43]:
# Check extraction results status (robust to out-of-order execution)
from pathlib import Path

results_dir = Path(GAO_RESULTS_DIR) if 'GAO_RESULTS_DIR' in globals() else None
chunks_dir = Path(GAO_CHUNKS_DIR) if 'GAO_CHUNKS_DIR' in globals() else None

if results_dir is not None and results_dir.exists():
    result_files = list(results_dir.glob('gao_*_chunk_*.json'))
    print(f'Extraction results found: {len(result_files)}')

    if len(result_files) > 0:
        # Derive chunk count directly so this cell does not depend on Cell 8 state
        chunk_count = None
        if chunks_dir is not None and chunks_dir.exists():
            chunk_count = len(list(chunks_dir.glob('gao_*_chunk_*.pdf')))
        coverage_denominator = chunk_count if chunk_count is not None else '?'
        print(f'Coverage: {len(result_files)}/{coverage_denominator} chunks')
else:
    print('No results directory found.')
    result_files = []

# Resolve prompt path robustly for actionable command output
if 'PROJECT_ROOT' in globals():
    prompt_path = Path(PROJECT_ROOT) / 'prompts' / 'gao_extraction_prompt.txt'
else:
    cwd = Path.cwd().resolve()
    prompt_path = cwd / 'prompts' / 'gao_extraction_prompt.txt'

print('\nTo run Gemini extraction:')
print('  1. Ensure GEMINI_API_KEY is set')
if prompt_path.exists():
    print(f'  2. Review/verify prompt: {prompt_path}')
else:
    print(f'  2. Prompt file not found at: {prompt_path}')
    print('     Confirm the prompt file path before running extraction.')

if 'PROJECT_ROOT' in globals():
    print(f'  3. Run: python {Path(PROJECT_ROOT) / "scripts" / "gemini_batch_extract.py"}')
else:
    print('  3. Run: python ../scripts/gemini_batch_extract.py')

Extraction results found: 22
Coverage: 22/22 chunks

To run Gemini extraction:
  1. Ensure GEMINI_API_KEY is set
  2. Review/verify prompt: C:\Users\sarme\IS392Final\IS392\prompts\gao_extraction_prompt.txt
  3. Run: python C:\Users\sarme\IS392Final\IS392\scripts\gemini_batch_extract.py


## 5. Step 4: Parse and Clean Results

In [ ]:
# Check if validation set exists
validation_path = Path(GAO_VALIDATION_PATH)

if validation_path.exists():
    print(f'✓ Validation set exists: {validation_path}')
    
    # Load and preview
    gao_df = pd.read_csv(validation_path)
    print(f'\nRecords: {len(gao_df)}')
    print(f'Columns: {list(gao_df.columns)}')

    if 'extraction_status' in gao_df.columns:
        print('\nExtraction status distribution:')
        print(gao_df['extraction_status'].fillna('missing').value_counts(dropna=False))

    tracked_cols = [col for col in ['gao_cost_growth', 'gao_schedule_delay', 'fpds_piid'] if col in gao_df.columns]
    if tracked_cols:
        print('\nMissing value counts:')
        print(gao_df[tracked_cols].isna().sum())

    print(f'\nSample:')
    print(gao_df.head())

    if 'extraction_status' in gao_df.columns and (gao_df['extraction_status'].fillna('') != 'ok').all():
        print('\nWarning: no usable GAO extraction rows are present yet.')
        print('Current records are placeholder/unparsed outputs, so downstream validation metrics would be misleading.')
else:
    print('Validation set not found.')
    print('\nTo parse results, run:')
    print('  python ../scripts/gemini_parse_results.py')

✓ Validation set exists: C:\Users\sarme\IS392Final\IS392\data\processed\gao_validation_set.csv

Records: 22
Columns: ['source_file', 'gao_program', 'gao_cost_growth', 'gao_schedule_delay', 'fpds_piid']

Sample:
                     source_file               gao_program  gao_cost_growth  \
0  gao_weapons_01_chunk_001.json  gao_weapons_01_chunk_001                0   
1  gao_weapons_02_chunk_001.json  gao_weapons_02_chunk_001                0   
2  gao_weapons_03_chunk_001.json  gao_weapons_03_chunk_001                0   
3  gao_weapons_04_chunk_001.json  gao_weapons_04_chunk_001                0   
4  gao_weapons_05_chunk_001.json  gao_weapons_05_chunk_001                0   

   gao_schedule_delay  fpds_piid  
0                   0        NaN  
1                   0        NaN  
2                   0        NaN  
3                   0        NaN  
4                   0        NaN  


## 6. Step 5: Link to FPDS

In [5]:
# Check if linked data exists (robust path resolution)
import pandas as pd
from pathlib import Path

if 'PROJECT_ROOT' in globals():
    notebook_root = Path(PROJECT_ROOT)
else:
    cwd = Path.cwd().resolve()
    notebook_root = None
    for base in [cwd] + list(cwd.parents):
        if (base / 'scripts').exists() and (base / 'data').exists():
            notebook_root = base
            break
        if (base / 'IS392' / 'scripts').exists() and (base / 'IS392' / 'data').exists():
            notebook_root = base / 'IS392'
            break
    if notebook_root is None:
        notebook_root = cwd

candidate_linked_paths = []
if 'GAO_FPDS_LINKED_PATH' in globals():
    candidate_linked_paths.append(Path(GAO_FPDS_LINKED_PATH))

candidate_linked_paths.extend([
    notebook_root / 'data' / 'processed' / 'gao_fpds_linked.csv',
    notebook_root.parent / 'IS392' / 'data' / 'processed' / 'gao_fpds_linked.csv' if notebook_root.name != 'IS392' else notebook_root / 'data' / 'processed' / 'gao_fpds_linked.csv',
])

# Deduplicate while preserving order
seen = set()
deduped = []
for p in candidate_linked_paths:
    rp = str(p.resolve())
    if rp not in seen:
        seen.add(rp)
        deduped.append(p)
candidate_linked_paths = deduped

linked_path = next((p for p in candidate_linked_paths if p.exists()), None)

if linked_path is not None:
    print(f'Linked data found: {linked_path}')
    
    # Load and preview
    linked_df = pd.read_csv(linked_path)
    print(f'\nLinked records: {len(linked_df)}')
    
    if len(linked_df) > 0:
        usable_df = linked_df
        if 'extraction_status' in linked_df.columns:
            status_counts = linked_df['extraction_status'].fillna('missing').value_counts(dropna=False)
            print('\nExtraction status distribution:')
            print(status_counts)
            
            usable_mask = linked_df['extraction_status'].fillna('') == 'ok'
            if not usable_mask.any():
                print('\nWarning: linked rows are based on placeholder or unparsed GAO extraction output.')
                print('Match stats below are not meaningful until real extraction results are generated.')
                usable_df = linked_df.iloc[0:0].copy()
            else:
                usable_df = linked_df.loc[usable_mask].copy()
                print(f'\nUsable linked records: {len(usable_df)}')
        
        preview_df = usable_df if len(usable_df) > 0 else linked_df
        
        if 'match_type' in preview_df.columns:
            print('\nMatch type distribution:')
            print(preview_df['match_type'].value_counts(dropna=False))
        
        if 'match_score' in preview_df.columns:
            print('\nMatch score stats:')
            print(preview_df['match_score'].describe())
        else:
            print('\nNote: match_type/match_score not in current linking format')
        
        print('\nSample linked record:')
        print(preview_df.iloc[0])
else:
    print('Linked data not found in expected locations.')
    print('\nChecked paths:')
    for p in candidate_linked_paths:
        print(f'  - {p}')

    processed_dir = notebook_root / 'data' / 'processed'
    if processed_dir.exists():
        nearby = sorted(processed_dir.glob('gao*linked*.csv'))
        if nearby:
            print('\nFound similar files:')
            for p in nearby:
                print(f'  - {p}')
    
    print('\nNote: Expect 20-40% match rate between GAO programs and FPDS contracts.')
    print('\nTo create linkages, run:')
    print('  pip install rapidfuzz  # if not installed')
    print(f'  python {notebook_root / "scripts" / "link_gao_to_fpds.py"}')

Linked data found: C:\Users\sarme\IS392Final\IS392\data\processed\gao_fpds_linked.csv

Linked records: 22

Extraction status distribution:
extraction_status
placeholder    22
Name: count, dtype: int64

Match stats below are not meaningful until real extraction results are generated.

Match type distribution:
match_type
none    22
Name: count, dtype: int64

Match score stats:
count    22.0
mean      0.0
std       0.0
min       0.0
25%       0.0
50%       0.0
75%       0.0
max       0.0
Name: match_score, dtype: float64

Sample linked record:
source_file                                   gao_weapons_01_chunk_001.json
source_chunk                                   gao_weapons_01_chunk_001.pdf
gao_program                                        gao_weapons_01_chunk_001
report_year                                                             NaN
service_branch                                                          NaN
baseline_cost_millions                                                  N

## 7. Step 6: Cross-Validate Classifier

In [24]:
# Run the real validation analysis (Claude-extracted data linked to FPDS).
# All heavy lifting lives in scripts/gao_validation_analysis.py so this
# cell is a thin wrapper that prints a full report and keeps the structured
# results in `results` for further ad-hoc analysis below.
import sys
from pathlib import Path
import importlib

# Resolve scripts directory robustly
if 'PROJECT_ROOT' in globals():
    scripts_path = Path(PROJECT_ROOT) / 'scripts'
else:
    cwd = Path.cwd().resolve()
    scripts_path = None
    for base in [cwd] + list(cwd.parents):
        if (base / 'scripts').exists():
            scripts_path = base / 'scripts'
            break
        if (base / 'IS392' / 'scripts').exists():
            scripts_path = base / 'IS392' / 'scripts'
            break
    if scripts_path is None:
        scripts_path = cwd / 'scripts'

scripts_path_str = str(scripts_path.resolve())
if scripts_path_str not in sys.path:
    sys.path.insert(0, scripts_path_str)

try:
    gao_module = importlib.import_module('gao_validation_analysis')
    results = gao_module.run_full_report()
except ModuleNotFoundError as e:
    print(f'Could not import gao_validation_analysis from {scripts_path_str}: {e}')
    print('Ensure scripts/gao_validation_analysis.py exists and rerun this cell.')
    results = None

Could not import gao_validation_analysis from C:\Users\sarme\IS392Final\IS392\scripts: No module named 'gao_validation_analysis'
Ensure scripts/gao_validation_analysis.py exists and rerun this cell.


## 8. Summary

In [6]:
# Final summary
import json
import pandas as pd
from pathlib import Path

if 'PROJECT_ROOT' in globals():
    summary_root = Path(PROJECT_ROOT)
else:
    cwd = Path.cwd().resolve()
    summary_root = None
    for base in [cwd] + list(cwd.parents):
        if (base / 'scripts').exists() and (base / 'data').exists():
            summary_root = base
            break
        if (base / 'IS392' / 'scripts').exists() and (base / 'IS392' / 'data').exists():
            summary_root = base / 'IS392'
            break
    if summary_root is None:
        summary_root = cwd

reports_dir = Path(GAO_REPORTS_DIR) if 'GAO_REPORTS_DIR' in globals() else summary_root / 'data' / 'raw' / 'gao_reports'
chunks_dir = Path(GAO_CHUNKS_DIR) if 'GAO_CHUNKS_DIR' in globals() else summary_root / 'data' / 'interim' / 'gao_chunks'
results_dir = Path(GAO_RESULTS_DIR) if 'GAO_RESULTS_DIR' in globals() else summary_root / 'data' / 'interim' / 'gao_results'
validation_path = Path(GAO_VALIDATION_PATH) if 'GAO_VALIDATION_PATH' in globals() else summary_root / 'data' / 'processed' / 'gao_validation_set.csv'
linked_path = Path(GAO_FPDS_LINKED_PATH) if 'GAO_FPDS_LINKED_PATH' in globals() else summary_root / 'data' / 'processed' / 'gao_fpds_linked.csv'

print('=' * 70)
print('GAO VALIDATION TRACK SUMMARY')
print('=' * 70)

status = []

# Check each step
if reports_dir.exists() and len(list(reports_dir.glob('*.pdf'))) >= 20:
    pdf_sizes = [p.stat().st_size for p in reports_dir.glob('*.pdf')]
    if pdf_sizes and min(pdf_sizes) <= 1024:
        status.append('⚠️ PDFs are placeholders - add real GAO reports before extraction')
    else:
        status.append('✓ PDFs downloaded (22 reports)')
else:
    status.append('⚠️ PDFs incomplete - run download_gao_pdfs.py')

if chunks_dir.exists() and len(list(chunks_dir.glob('*.pdf'))) > 0:
    chunks = len(list(chunks_dir.glob('*.pdf')))
    status.append(f'✓ PDFs chunked ({chunks} chunks)')
else:
    status.append('⚠️ Chunking needed - run chunk_gao_pdfs.py')

if results_dir.exists() and len(list(results_dir.glob('*.json'))) > 0:
    results = len(list(results_dir.glob('*.json')))
    placeholder_results = 0
    for result_path in results_dir.glob('*.json'):
        try:
            with open(result_path, 'r', encoding='utf-8') as handle:
                result_payload = json.load(handle)
            if isinstance(result_payload, dict):
                if str(result_payload.get('extraction_status', '')).lower() != 'ok':
                    placeholder_results += 1
            elif not isinstance(result_payload, list):
                placeholder_results += 1
        except (OSError, ValueError, TypeError):
            placeholder_results += 1
    if placeholder_results == results:
        status.append(f'⚠️ Extraction outputs are placeholders ({results} files)')
    else:
        status.append(f'✓ Gemini extraction complete ({results - placeholder_results} usable results, {placeholder_results} placeholders)')
else:
    status.append('⚠️ Extraction needed - run gemini_batch_extract.py')

if validation_path.exists():
    gao_df = pd.read_csv(validation_path)
    if 'extraction_status' in gao_df.columns and (gao_df['extraction_status'].fillna('') != 'ok').all():
        status.append(f'⚠️ Validation set parsed but all {len(gao_df)} rows are placeholders')
    else:
        usable_rows = int((gao_df['extraction_status'].fillna('') == 'ok').sum()) if 'extraction_status' in gao_df.columns else len(gao_df)
        status.append(f'✓ Validation set parsed ({usable_rows} usable programs)')
else:
    status.append('⚠️ Parsing needed - run gemini_parse_results.py')

if linked_path.exists():
    linked_df = pd.read_csv(linked_path)
    if 'extraction_status' in linked_df.columns and (linked_df['extraction_status'].fillna('') != 'ok').all():
        status.append(f'⚠️ Linked file exists but all {len(linked_df)} rows come from placeholder extraction')
    else:
        usable_linked = int((linked_df['extraction_status'].fillna('') == 'ok').sum()) if 'extraction_status' in linked_df.columns else len(linked_df)
        status.append(f'✓ Linked to FPDS ({usable_linked} usable matches)')
else:
    status.append('⚠️ Linking needed - run link_gao_to_fpds.py')

print('\nStatus:')
for s in status:
    print(f'  {s}')

print('\n' + '=' * 70)
print('Next Steps:')
print('  1. Replace placeholder GAO PDFs with real report URLs/files')
print('  2. Run extraction, parse, and linking again')
print('  3. Review cross-validation results only after usable GAO rows exist')
print('=' * 70)

GAO VALIDATION TRACK SUMMARY

Status:
  ⚠️ PDFs are placeholders - add real GAO reports before extraction
  ✓ PDFs chunked (22 chunks)
  ⚠️ Extraction outputs are placeholders (22 files)
  ⚠️ Validation set parsed but all 22 rows are placeholders
  ⚠️ Linked file exists but all 22 rows come from placeholder extraction

Next Steps:
  1. Replace placeholder GAO PDFs with real report URLs/files
  2. Run extraction, parse, and linking again
  3. Review cross-validation results only after usable GAO rows exist
